# 03_indice_oportunidad

## Objetivo
Calcular el índice de oportunidad de microcrédito digital por departamento a partir del dataset maestro.

## Alcance
- Cargar `master_dataset.csv`
- Normalizar variables
- Definir pesos
- Calcular índice
- Generar ranking
- Explorar escenarios (opcional)

## 1. Librerías

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

## 2. Rutas y carga de datos

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

df = pd.read_csv(DATA_PROCESSED / 'master_dataset.csv')
df.head()

## 3. Revisión inicial

In [ ]:
print(df.shape)
print(df.isna().sum())
df.describe()

## 4. Normalización de variables
Se utiliza Min-Max scaling para llevar todas las variables al rango [0,1].

In [ ]:
def minmax(series):
    return (series - series.min()) / (series.max() - series.min())

df_norm = df.copy()

df_norm['pobreza_n'] = minmax(df['pobreza_2024'])
df_norm['microcredito_n'] = 1 - minmax(df['acceso_microcredito_2024'])
df_norm['productos_n'] = 1 - minmax(df['acceso_productos_financieros_2024'])
df_norm['atm_n'] = 1 - minmax(df['atm_x_10000_adultos_2024'])
df_norm['internet_n'] = minmax(df['internet_hogares_2024'])

df_norm[['departamento','pobreza_n','microcredito_n','productos_n','atm_n','internet_n']].head()

## 5. Definición de pesos
Los pesos representan la importancia relativa de cada dimensión.

In [ ]:
pesos = {
    'pobreza_n': 0.30,
    'microcredito_n': 0.30,
    'productos_n': 0.15,
    'atm_n': 0.10,
    'internet_n': 0.15
}

pesos

## 6. Cálculo del índice de oportunidad

In [ ]:
df_norm['indice_oportunidad'] = (
    df_norm['pobreza_n'] * pesos['pobreza_n'] +
    df_norm['microcredito_n'] * pesos['microcredito_n'] +
    df_norm['productos_n'] * pesos['productos_n'] +
    df_norm['atm_n'] * pesos['atm_n'] +
    df_norm['internet_n'] * pesos['internet_n']
)

df_norm[['departamento','indice_oportunidad']].head()

## 7. Ranking de departamentos

In [ ]:
ranking = df_norm.sort_values(by='indice_oportunidad', ascending=False).reset_index(drop=True)
ranking[['departamento','indice_oportunidad']].head(10)

## 8. Clasificación por niveles

In [ ]:
df_norm['nivel'] = pd.qcut(df_norm['indice_oportunidad'], q=3, labels=['Bajo','Medio','Alto'])
df_norm[['departamento','indice_oportunidad','nivel']].sort_values(by='indice_oportunidad', ascending=False).head(10)

## 9. Guardar resultados

In [ ]:
output = DATA_PROCESSED / 'ranking_oportunidad.csv'
df_norm.to_csv(output, index=False)
print('Guardado en:', output)

## 10. Exploración de escenarios (opcional)
Permite modificar pesos para observar cambios en el ranking.

In [ ]:
# ejemplo de escenario alternativo
pesos_alt = pesos.copy()
pesos_alt['internet_n'] = 0.30
pesos_alt['pobreza_n'] = 0.20

df_norm['indice_alt'] = (
    df_norm['pobreza_n'] * pesos_alt['pobreza_n'] +
    df_norm['microcredito_n'] * pesos_alt['microcredito_n'] +
    df_norm['productos_n'] * pesos_alt['productos_n'] +
    df_norm['atm_n'] * pesos_alt['atm_n'] +
    df_norm['internet_n'] * pesos_alt['internet_n']
)

df_norm.sort_values(by='indice_alt', ascending=False)[['departamento','indice_alt']].head(10)